In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
#  Load Dataset
spark = (
    SparkSession.builder
    .appName("ElectricityTheftDetection")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
    )


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/12 11:45:39 WARN Utils: Your hostname, sandeeps-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.16.247.72 instead (on interface en0)
26/10/12 11:45:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/saibhor/Desktop/ELEECTRICITY_THEFT_DETECTION/venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/12 11:45:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


Rows: 42372
Columns: 1036
root
 |-- CONS_NO: string (nullable = true)
 |-- FLAG: integer (nullable = true)
 |-- 2014/1/1: double (nullable = true)
 |-- 2014/1/10: double (nullable = true)
 |-- 2014/1/11: double (nullable = true)
 |-- 2014/1/12: double (nullable = true)
 |-- 2014/1/13: double (nullable = true)
 |-- 2014/1/14: double (nullable = true)
 |-- 2014/1/15: double (nullable = true)
 |-- 2014/1/16: double (nullable = true)
 |-- 2014/1/17: double (nullable = true)
 |-- 2014/1/18: double (nullable = true)
 |-- 2014/1/19: double (nullable = true)
 |-- 2014/1/2: double (nullable = true)
 |-- 2014/1/20: double (nullable = true)
 |-- 2014/1/21: double (nullable = true)
 |-- 2014/1/22: double (nullable = true)
 |-- 2014/1/23: double (nullable = true)
 |-- 2014/1/24: double (nullable = true)
 |-- 2014/1/25: double (nullable = true)
 |-- 2014/1/26: double (nullable = true)
 |-- 2014/1/27: double (nullable = true)
 |-- 2014/1/28: double (nullable = true)
 |-- 2014/1/29: double (nullable =

26/10/12 11:46:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------+----+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------+---------+---------+--------+--------+--------+--------+--------+--------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+---------+---------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+---------+---------+---------+---------+---------+---------+-------

In [ ]:
data_path = "../data/raw/data.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode","PERMISSIVE")
    .csv(data_path)
)

print("Spark version:", spark.version)
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

In [4]:
print(df.count())
print(len(df.columns))
df.groupBy("FLAG").count().show()

42372
1036


+----+-----+
|FLAG|count|
+----+-----+
|   1| 3615|
|   0|38757|
+----+-----+



In [5]:
print("columns",df.columns[0:])
df.select(df.columns[0], "FLAG").show(5)

columns ['CONS_NO', 'FLAG', '2014/1/1', '2014/1/10', '2014/1/11', '2014/1/12', '2014/1/13', '2014/1/14', '2014/1/15', '2014/1/16', '2014/1/17', '2014/1/18', '2014/1/19', '2014/1/2', '2014/1/20', '2014/1/21', '2014/1/22', '2014/1/23', '2014/1/24', '2014/1/25', '2014/1/26', '2014/1/27', '2014/1/28', '2014/1/29', '2014/1/3', '2014/1/30', '2014/1/31', '2014/1/4', '2014/1/5', '2014/1/6', '2014/1/7', '2014/1/8', '2014/1/9', '2014/10/1', '2014/10/10', '2014/10/11', '2014/10/12', '2014/10/13', '2014/10/14', '2014/10/15', '2014/10/16', '2014/10/17', '2014/10/18', '2014/10/19', '2014/10/2', '2014/10/20', '2014/10/21', '2014/10/22', '2014/10/23', '2014/10/24', '2014/10/25', '2014/10/26', '2014/10/27', '2014/10/28', '2014/10/29', '2014/10/3', '2014/10/30', '2014/10/31', '2014/10/4', '2014/10/5', '2014/10/6', '2014/10/7', '2014/10/8', '2014/10/9', '2014/11/1', '2014/11/10', '2014/11/11', '2014/11/12', '2014/11/13', '2014/11/14', '2014/11/15', '2014/11/16', '2014/11/17', '2014/11/18', '2014/11/19', 

In [8]:

reading_cols = [
    c for c in df.columns
    if c not in ["CONS_NO", "FLAG"]
]

# Convert reading columns to numeric values
df_numeric = df.select(
    F.col("CONS_NO"),
    F.col("FLAG").cast("int").alias("FLAG"),
    *[
        F.col(c).cast("double").alias(c)
        for c in reading_cols
    ]
)

# Create an array of daily readings for each customer
readings_array = F.array(*[F.col(c) for c in reading_cols])

# Ignore null readings when calculating statistics
feature_df = df_numeric.select(
    "CONS_NO",
    "FLAG",
    F.expr(
        "aggregate("
        + "filter(array("
        + ",".join([f"`{c}`" for c in reading_cols])
        + "), x -> x is not null), "
        + "cast(0 as double), "
        + "(acc, x) -> acc + x)"
    ).alias("total_consumption"),
    F.expr(
        "aggregate("
        + "filter(array("
        + ",".join([f"`{c}`" for c in reading_cols])
        + "), x -> x is not null), "
        + "cast(0 as double), "
        + "(acc, x) -> acc + x)"
    ).alias("consumption_sum"),
    F.size(
        F.filter(readings_array, lambda x: x.isNotNull())
    ).alias("valid_readings")
)

feature_df = (
    feature_df
    .withColumn(
        "missing_ratio",
        F.lit(len(reading_cols) - F.col("valid_readings"))
        / F.lit(len(reading_cols))
    )
    .withColumn(
        "mean_consumption",
        F.when(
            F.col("valid_readings") > 0,
            F.col("consumption_sum") / F.col("valid_readings")
        )
    )
)

feature_df.select(
    "CONS_NO", "FLAG", "valid_readings",
    "missing_ratio", "mean_consumption", "total_consumption"
).show(5, truncate=False)


+--------------------------------+----+--------------+---------------------+------------------+-----------------+
|CONS_NO                         |FLAG|valid_readings|missing_ratio        |mean_consumption  |total_consumption|
+--------------------------------+----+--------------+---------------------+------------------+-----------------+
|0387DD8A07E07FDA6271170F86AD9151|1   |591           |0.4284332688588008   |13.89834179357021 |8213.919999999995|
|01D6177B5D4FFE0CABA9EF17DAFC2B84|1   |304           |0.7059961315280464   |0.0               |0.0              |
|4B75AC4F2D8434CFF62DB64D0BB43103|1   |259           |0.7495164410058027   |9.921389961389965 |2569.640000000001|
|B32AC8CC6D5D805AC053557AB05F5343|1   |472           |0.5435203094777563   |14.612055084745764|6896.89          |
|EDFC78B07BA2908B3395C4EB2304665E|1   |1030          |0.0038684719535783366|8.249737864077677 |8497.230000000007|
+--------------------------------+----+--------------+---------------------+------------

In [12]:

# Remove the duplicate column and keep total_consumption
customer_features = feature_df.drop("consumption_sum")

print("Updated columns:", customer_features.columns)


Updated columns: ['CONS_NO', 'FLAG', 'total_consumption', 'valid_readings', 'missing_ratio', 'mean_consumption']


In [16]:

# Define the output directory
output_path = "../data/processed/customer_features"

# Save the customer-level features in Parquet format
(
    customer_features.write
    .mode("overwrite")
    .parquet(output_path)
)

print("Customer features saved successfully!")
print("Saved to:", output_path)


Customer features saved successfully!
Saved to: ../data/processed/customer_features


In [17]:

# Reload the saved Parquet data

features_check = spark.read.parquet(output_path)

print("Columns:", features_check.columns)
print("Number of columns:", len(features_check.columns))
print("Number of rows:", features_check.count())

features_check.show(5, truncate=False)



Columns: ['CONS_NO', 'FLAG', 'total_consumption', 'valid_readings', 'missing_ratio', 'mean_consumption']
Number of columns: 6
Number of rows: 42372
+--------------------------------+----+-----------------+--------------+---------------------+------------------+
|CONS_NO                         |FLAG|total_consumption|valid_readings|missing_ratio        |mean_consumption  |
+--------------------------------+----+-----------------+--------------+---------------------+------------------+
|0387DD8A07E07FDA6271170F86AD9151|1   |8213.919999999995|591           |0.4284332688588008   |13.89834179357021 |
|01D6177B5D4FFE0CABA9EF17DAFC2B84|1   |0.0              |304           |0.7059961315280464   |0.0               |
|4B75AC4F2D8434CFF62DB64D0BB43103|1   |2569.640000000001|259           |0.7495164410058027   |9.921389961389965 |
|B32AC8CC6D5D805AC053557AB05F5343|1   |6896.89          |472           |0.5435203094777563   |14.612055084745764|
|EDFC78B07BA2908B3395C4EB2304665E|1   |8497.2300000000

In [18]:

# Count customers in each target class
class_distribution = (
    features_check
    .groupBy("FLAG")
    .count()
    .orderBy("FLAG")
)

class_distribution.show()

# Calculate class percentages
total_customers = features_check.count()

class_distribution.withColumn(
    "percentage",
    F.round(F.col("count") / total_customers * 100, 2)
).show()


+----+-----+
|FLAG|count|
+----+-----+
|   0|38757|
|   1| 3615|
+----+-----+

+----+-----+----------+
|FLAG|count|percentage|
+----+-----+----------+
|   0|38757|     91.47|
|   1| 3615|      8.53|
+----+-----+----------+



In [19]:

# Summary statistics for the numerical features
features_check.select(
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
).describe().show()


+-------+-----------------+--------------------+-----------------+-----------------+
|summary|   valid_readings|       missing_ratio| mean_consumption|total_consumption|
+-------+-----------------+--------------------+-----------------+-----------------+
|  count|            42372|               42372|            42367|            42372|
|   mean|768.8832247710752| 0.25639920234905755|9.944193224266936|7118.982204797967|
| stddev|298.0590564119112|  0.2882582750598752|82.03726142691521| 34705.2009015358|
|    min|                0|9.671179883945841E-4|              0.0|              0.0|
|    max|             1033|                 1.0|          13601.1|        4080330.0|
+-------+-----------------+--------------------+-----------------+-----------------+



In [20]:

# Compare average feature values for each FLAG class
features_check.groupBy("FLAG").agg(
    F.count("*").alias("customer_count"),
    F.round(F.avg("mean_consumption"), 3).alias("avg_mean_consumption"),
    F.round(F.avg("total_consumption"), 3).alias("avg_total_consumption"),
    F.round(F.avg("missing_ratio"), 3).alias("avg_missing_ratio"),
    F.round(F.avg("valid_readings"), 1).alias("avg_valid_readings")
).orderBy("FLAG").show()


+----+--------------+--------------------+---------------------+-----------------+------------------+
|FLAG|customer_count|avg_mean_consumption|avg_total_consumption|avg_missing_ratio|avg_valid_readings|
+----+--------------+--------------------+---------------------+-----------------+------------------+
|   0|         38757|               7.949|             5963.345|            0.251|             774.5|
|   1|          3615|              31.348|            19508.759|            0.315|             708.6|
+----+--------------+--------------------+---------------------+-----------------+------------------+



In [21]:

from pyspark.sql import functions as F

# 1. Check missing values in each column
missing_summary = features_check.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in features_check.columns
])

missing_summary.show(truncate=False)

# 2. Check duplicate customer IDs
duplicate_ids = (
    features_check
    .groupBy("CONS_NO")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_ids.count())

# 3. Check invalid values in the engineered features
features_check.select(
    F.sum(
        F.when(F.col("valid_readings") < 0, 1).otherwise(0)
    ).alias("negative_valid_readings"),

    F.sum(
        F.when(
            (F.col("missing_ratio") < 0) |
            (F.col("missing_ratio") > 1),
            1
        ).otherwise(0)
    ).alias("invalid_missing_ratio"),

    F.sum(
        F.when(
            F.col("mean_consumption") < 0, 1
        ).otherwise(0)
    ).alias("negative_mean_consumption"),

    F.sum(
        F.when(
            F.col("total_consumption") < 0, 1
        ).otherwise(0)
    ).alias("negative_total_consumption")
).show()



+-------+----+-----------------+--------------+-------------+----------------+
|CONS_NO|FLAG|total_consumption|valid_readings|missing_ratio|mean_consumption|
+-------+----+-----------------+--------------+-------------+----------------+
|0      |0   |0                |0             |0            |5               |
+-------+----+-----------------+--------------+-------------+----------------+

Duplicate customer IDs: 0
+-----------------------+---------------------+-------------------------+--------------------------+
|negative_valid_readings|invalid_missing_ratio|negative_mean_consumption|negative_total_consumption|
+-----------------------+---------------------+-------------------------+--------------------------+
|                      0|                    0|                        0|                         0|
+-----------------------+---------------------+-------------------------+--------------------------+



In [22]:
model_data = features_check.select(
    "FLAG",
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
)

# stratified split
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
print("training rows:", train_data.count())
print("testing row: ", test_data.count())
print("trainingclass", train_data.groupBy("FLAG").count().orderBy("FLAG").show())
print("testingclass", test_data.groupBy("FLAG").count().orderBy("FLAG").show())

training rows: 34074
testing row:  8298
+----+-----+
|FLAG|count|
+----+-----+
|   0|31176|
|   1| 2898|
+----+-----+

trainingclass None
+----+-----+
|FLAG|count|
+----+-----+
|   0| 7581|
|   1|  717|
+----+-----+

testingclass None


26/10/12 15:34:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 133930 ms exceeds timeout 120000 ms
26/10/12 15:34:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/12 15:34:07 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Numerical inputs; CONS_NO is intentionally excluded
feature_cols = [
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
]

# Calculate class weights from the training set
class_counts = {
    row["FLAG"]: row["count"]
    for row in train_data.groupBy("FLAG").count().collect()
}

count_0 = class_counts.get(0, 0)
count_1 = class_counts.get(1, 0)
total_train = count_0 + count_1

if count_0 == 0 or count_1 == 0:
    raise ValueError("Training data must contain both target classes.")

weight_0 = total_train / (2.0 * count_0)
weight_1 = total_train / (2.0 * count_1)

weighted_train = train_data.withColumn(
    "class_weight",
    F.when(F.col("FLAG") == 1, F.lit(weight_1))
     .otherwise(F.lit(weight_0))
)

print("Class weights:", {0: weight_0, 1: weight_1})

# 1. Impute missing numerical features using training medians
imputer = Imputer(
    inputCols=feature_cols,
    outputCols=[c + "_imputed" for c in feature_cols],
    strategy="median"
)

imputed_cols = [c + "_imputed" for c in feature_cols]

# 2. Assemble the numerical inputs into one vector
assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features"
)

# 3. Define the baseline classifier
lr = LogisticRegression(
    featuresCol="features",
    labelCol="FLAG",
    weightCol="class_weight",
    maxIter=50,
    regParam=0.01
)

# 4. Create the pipeline
pipeline = Pipeline(stages=[imputer, assembler, lr])

# 5. Fit only on training data
lr_model = pipeline.fit(weighted_train)

print("Logistic Regression training completed!")
